# Лабораторная работа №3. Циклы и автоматизация: Z-оценка и скользящее среднее

---

## 1. Цель работы

### Обучающая цель

Освоить цикл `for`, функции `range()`, `enumerate()`, `break`, `continue`, списковые включения. Реализовать вручную дисперсию и стандартное отклонение (σ), Z-оценку и скользящее среднее. Научиться автоматически находить аномалии в выборке.

### Профессиональная цель

Научиться автоматизированному скринингу транзакций: программа проходит по всей выписке, накапливает статистики, находит аномалии и выявляет тренды. Понять, чем Z-оценка отличается от метода IQR, и уметь объяснить, зачем нужно скользящее среднее.

### Связь со статистикой

Дисперсия, стандартное отклонение ($\sigma$), Z-оценка, правило трёх сигм, скользящее среднее, накопление статистик, фильтрация.

---

## 2. Планируемые результаты

**Знаем:**
- Что такое цикл `for` и чем он отличается от `while`.
- Как работает `range()` — три формы записи.
- Что делает `enumerate()` и зачем он нужен.
- Что делают `break` и `continue`.
- Что такое списковые включения.
- Что такое дисперсия и стандартное отклонение ($\sigma$) — корень из дисперсии.
- Что такое Z-оценка и правило трёх сигм.
- Что такое скользящее среднее и зачем оно нужно.

**Умеем:**
- Использовать `for` для перебора элементов списка.
- Применять `range()` для перебора индексов.
- Использовать `enumerate()` для получения индекса и значения одновременно.
- Прерывать цикл через `break` и пропускать итерацию через `continue`.
- Строить новые списки через списковые включения (list comrehensions).
- Вычислять дисперсию, $\sigma$ и Z-оценку вручную.
- Находить аномалии двумя методами: IQR и Z-оценка.
- Строить скользящее среднее для выявления трендов.

**Владеем:**
- Навыком автоматизированной обработки выписок.
- Навыком сравнивать методы поиска аномалий.
- Навыком формулировать выводы по результатам вычислений.

---

## 3. Теоретический материал

### 3.1. Зачем нужны циклы

В лабораторной №2 мы работали с одной выборкой из 20 транзакций. Но в реальной работе безопасник имеет дело с **тысячами** транзакций за месяц. Обрабатывать их вручную невозможно.

> **Цикл** — это конструкция, которая позволяет повторять одни и те же действия для каждого элемента выборки.

В Python для перебора элементов используются два цикла: `while` (из лабораторной №1) и `for`. Цикл `for` — основной инструмент для работы с выборками.

В этой лабораторной мы:
- освоим цикл `for`, `range()`, `enumerate()`;
- научимся использовать `break` и `continue`;
- построим списковые включения;
- реализуем вручную дисперсию, стандартное отклонение ($\sigma$) и Z-оценку;
- сравним два метода поиска аномалий: IQR и Z-оценка;
- построим скользящее среднее.

### 3.2. Цикл `for`

In [ ]:
for элемент in последовательность:
    # действия для каждого элемента

In [ ]:
## Пример:
amounts = [5000, 12000, 8000]
for a in amounts:
    print(a)

Переменная `a` называется **счётчиком цикла**. Имя может быть любым — главное, чтобы оно было осмысленным.

### 3.3. Функция `range()`

`range()` — это функция, которая генерирует последовательность чисел. Три формы записи:

| Форма | Что делает | Пример |
|-------|------------|--------|
| `range(stop)` | 0, 1, 2, …, stop−1 | `range(5)` → 0, 1, 2, 3, 4 |
| `range(start, stop)` | start, …, stop−1 | `range(2, 6)` → 2, 3, 4, 5 |
| `range(start, stop, step)` | с шагом step | `range(0, 10, 2)` → 0, 2, 4, 6, 8 |

**Правый конец не включается.**

In [ ]:
for i in range(5):
    print(i)   # проверьте вывод → 0 1 2 3 4

In [ ]:
for i in range(2, 6):
    print(i)   # проверьте вывод  → 2 3 4 5

In [ ]:
for i in range(0, 10, 2):
    print(i)   # проверьте вывод  → 0 2 4 6 8

**Полезный трюк:** `range(len(amounts))` даёт все индексы списка.

In [ ]:
amounts = [5000, 12000, 8000]
for i in range(len(amounts)):
    print(i, amounts[i])

### 3.4. Функция `enumerate()`

Если нужны **и индекс, и значение**, удобнее использовать `enumerate()`.

In [ ]:
amounts = [5000, 12000, 8000]
for i, a in enumerate(amounts):
    print(i, a)

### 3.5. `break` и `continue`

**`break`** — прерывает цикл полностью:

In [ ]:
amounts = [5000, 12000, 300000, 8000]
for a in amounts:
    if a > 100000:
        print(f"Первая крупная транзакция: {a}")
        break   # прерывает цикл, когда находит первую транзакцию крупнее 100000

**`continue`** — пропускает текущую итерацию и переходит к следующей:

In [ ]:
amounts = [5000, 12000, 300000, 8000]
for a in amounts:
    if a < 10000:
        continue
    print(f"Крупная транзакция: {a}")

### 3.6. Списковые включения

**Списковое включение** — это компактный способ создать новый список на основе старого.

**Обычный цикл:**

In [ ]:
amounts = [5000, 12000, 8000]
doubled = []
for a in amounts:
    doubled.append(a * 2)

**Через списковое включение:**

In [ ]:
doubled = [a * 2 for a in amounts]
## короче, pythonистее и изящнее

**С условием:**

In [ ]:
large = [a for a in amounts if a > 10000]
## оптимизирует цикл и условие if одновременно

**С преобразованием и условием:**

In [ ]:
large_doubled = [a * 2 for a in amounts if a > 10000]
## оптимизирует цикл, условие if и преобразование значений

### 3.7. Дисперсия

**Дисперсия** — это мера разброса данных относительно среднего. Показывает, насколько в среднем значения отклоняются от среднего, но в квадрате.

**Формула выборочной дисперсии:**

$$\sigma^2 = \frac{\sum_{i=1}^{n}(x_i - \bar{x})^2}{n - 1}$$

**Алгоритм:**
1. Вычислить среднее $\bar{x}$.
2. Для каждого элемента найти отклонение от среднего.
3. Возвести отклонения в квадрат.
4. Сложить квадраты.
5. Разделить на $(n - 1)$.

**Реализация в Python:**

In [ ]:
amounts = [5000, 12000, 8000, 15000, 7000]
n = len(amounts)
mean = sum(amounts) / n

diffs = 0
for a in amounts:
    diffs += (a - mean) ** 2

variance = diffs / (n - 1)
print(f"Дисперсия: {variance:.2f}")

### 3.8. Стандартное отклонение ($\sigma$)

**Стандартное отклонение** обозначается греческой буквой **$\sigma$** (сигма) и представляет собой **корень из дисперсии**. Оно показывает, насколько в среднем значения отклоняются от среднего в тех же единицах, что и сами данные.

**Формула:**

$$\sigma = \sqrt{\sigma^2} = \sqrt{\frac{\sum_{i=1}^{n}(x_i - \bar{x})^2}{n - 1}}$$

**Реализация в Python:**

In [ ]:
import math

sigma = math.sqrt(variance)
print(f"Стандартное отклонение: {sigma:.2f}")

In [ ]:
## Или сразу из суммы квадратов:


amounts = [5000, 12000, 8000, 15000, 7000]
n = len(amounts)
mean = sum(amounts) / n

diffs = 0
for a in amounts:
    diffs += (a - mean) ** 2

sigma = math.sqrt(diffs / (n - 1))
print(f"σ = {sigma:.2f}")

### 3.9. Z-оценка

**Z-оценка** (стандартизованное значение) показывает, на сколько стандартных отклонений значение отличается от среднего.

**Формула:**

$$z_i = \frac{x_i - \bar{x}}{\sigma}$$

**Интерпретация:**

| Значение Z | Интерпретация |
|------------|---------------|
| \|Z\| < 2 | Типичное значение |
| 2 ≤ \|Z\| < 3 | Подозрительное значение |
| \|Z\| ≥ 3 | Аномалия (правило трёх сигм) |

**Реализация:**

In [ ]:
z_scores = []
for a in amounts:
    z = (a - mean) / sigma
    z_scores.append(z)

for i, z in enumerate(z_scores):
    if abs(z) >= 3:
        print(f"Аномалия: {amounts[i]} (Z = {z:.2f})")

### 3.10. Правило трёх сигм

Если данные распределены нормально, то:

- **68 %** значений лежат в пределах **±1** **$\sigma$** от среднего;
- **95 %** значений лежат в пределах **±2** **$\sigma$**;
- **99,7 %** значений лежат в пределах **±3** **$\sigma$**.

Всё, что выходит за пределы **±3** **$\sigma$**, считается аномалией.

In [ ]:
lower = mean - 3 * sigma
upper = mean + 3 * sigma

for a in amounts:
    if a < lower or a > upper:
        print(f"Аномалия: {a}")

### 3.11. Скользящее среднее

**Скользящее среднее** — это среднее значение за последние `k` наблюдений. Помогает сгладить случайные колебания и увидеть тренд.

**Алгоритм:**
1. Задать размер окна `k`.
2. Для каждой позиции `i` (начиная с `k-1`) взять последние `k` элементов.
3. Вычислить их среднее.
4. Записать результат.

**Реализация:**

In [ ]:
amounts = [5000, 5200, 5100, 5300, 5500, 5400, 5600]
k = 3
ma = []

for i in range(k - 1, len(amounts)):
    window = amounts[i - k + 1 : i + 1]
    ma.append(sum(window) / k)

print(ma)   # → [5100.0, 5200.0, 5300.0, 5400.0, 5500.0]

### 3.12. Сравнение методов: IQR и Z-оценка

Оба метода находят аномалии, но по-разному:

| Метод | Основа | Устойчивость к выбросам | Когда использовать |
|-------|--------|-------------------------|---------------------|
| **IQR** | Квартили | Устойчив | Всегда, особенно для скошенных данных |
| **Z-оценка** | Среднее и $\sigma$ | Не устойчив (сам зависит от выбросов) | Для нормально распределённых данных |

**Вывод:** если данных мало или они сильно скошены, IQR надёжнее. Если данных много и они распределены нормально — Z-оценка работает хорошо.

## 4. Ход работы

**Общее время:**  2 пары.

**Структура:**

| № | Задание 
|---|---------
| 1 | Цикл `for` и `range()` 
| 2 | `enumerate()`, `break`, `continue` 
| 3 | Списковые включения 
| 4 | Дисперсия и стандартное отклонение ($\sigma$) 
| 5 | Z-оценка и правило трёх сигм 
| 6 | Скользящее среднее 
| 7 | Профильный кейс: автоматизированный скрининг 
| 8 | Вывод  

**Рекомендация**
Реализацию и разбор всех заданий смотрите выше, в теоретической части.

### Задание 1. Цикл `for` и `range()` 

🎯 **Цель:** освоить цикл `for`, функцию `range()`, перебор по индексам.

#### 1.1. Простой перебор

✏️ **Задание:** создайте список `amounts = [3000, 7000, 12000, 5000, 9000]`. Выведите каждый элемент на отдельной строке с помощью цикла `for`.

In [ ]:
## код пишите здесь

#### 1.2. Сумма через цикл

✏️ **Задание:** вычислите сумму элементов списка `amounts` с помощью цикла `for`, не используя встроенную функцию `sum()`. Выведите результат.

In [ ]:
## код пишите здесь

#### 1.3. Перебор по индексам через `range()`

✏️ **Задание:** выведите на экран индекс и значение каждого элемента списка `amounts` в формате `«Индекс 0: 3000»`. Используйте `range(len(amounts))`.


In [ ]:
## код пишите здесь

#### 1.4. `range()` с шагом

✏️ **Задание:** выведите на экран все чётные числа от 0 до 20, используя `range()` с шагом 2.

In [ ]:
## код пишите здесь

#### 1.5. Обратный `range()`

✏️ **Задание:** выведите на экран числа от 10 до 1 в обратном порядке, используя `range()` с отрицательным шагом.

🔍 **Вопрос для самопроверки:** почему `range(5)` не включает 5? Что произойдёт, если написать `range(len(amounts) + 1)`?

In [ ]:
## код пишите здесь

### Задание 2. `enumerate()`, `break`, `continue` 

🎯 **Цель:** освоить `enumerate()`, прерывание и пропуск итераций.

#### 2.1. `enumerate()`

✏️ **Задание:** для списка `amounts = [3000, 7000, 12000, 5000, 9000]` выведите индекс и значение каждого элемента в формате `«Позиция 0 → 3000»`. Используйте `enumerate()`.

In [ ]:
## код пишите здесь

#### 2.2. `break` — поиск первой крупной транзакции

✏️ **Задание:** для списка `amounts = [4500, 7200, 12000, 35000, 8000, 51000]` найдите первую транзакцию, которая превышает 30 000. Выведите её значение и индекс. Прервите цикл с помощью `break`.

In [ ]:
## код пишите здесь

#### 2.3. `continue` — пропуск мелких платежей

✏️ **Задание:** для списка `amounts = [3000, 15000, 2500, 8000, 12000, 1000]` выведите только те транзакции, которые больше 5 000. Используйте `continue`, чтобы пропустить мелкие платежи.

In [ ]:
## код пишите здесь

#### 2.4. Комбинация `break` и `continue`

✏️ **Задание:** для списка `amounts = [3000, 25000, 4000, 50000, 2000, 300000, 1000]`:
- пропустите транзакции меньше 5 000 (через `continue`);
- выведите транзакции от 5 000 до 100 000;
- прервите цикл, если встретится транзакция больше 100 000 (через `break`).

🔍 **Вопрос для самопроверки:** в чём разница между `break` и `continue`? Что произойдёт, если поменять их местами?

In [ ]:
## код пишите здесь

### Задание 3. Списковые включения 
🎯 **Цель:** освоить списковые включения как компактный способ создания списков.

#### 3.1. Простое включение

✏️ **Задание:** для списка `amounts = [3000, 7000, 12000, 5000, 9000]` создайте новый список `doubled`, в котором каждый элемент увеличен в 2 раза. Используйте списковое включение.

In [ ]:
## код пишите здесь

#### 3.2. Включение с условием

✏️ **Задание:** создайте список `large`, в который попадут только транзакции больше 6 000.

In [ ]:
## код пишите здесь

#### 3.3. Включение с преобразованием и условием

✏️ **Задание:** создайте список `percent`, в который попадут 10 % от каждой транзакции, превышающей 5 000.

In [ ]:
## код пишите здесь

#### 3.4. Включение для фильтрации выбросов

✏️ **Задание:** для списка `amounts = [5000, 7000, 500000, 8000, 12000]` создайте список `clean` без выброса (500 000). Используйте условие в списковом включении.

In [ ]:
## код пишите здесь

#### 3.5. Сравнение с обычным циклом

✏️ **Задание:** решите задачу 3.2 двумя способами: через обычный цикл `for` и через списковое включение. Сравните длину кода.

🔍 **Вопрос для самопроверки:** когда списковое включение удобнее обычного цикла? Когда — наоборот?

In [ ]:
## код пишите здесь

### Задание 4. Дисперсия и стандартное отклонение ($\sigma$) 

🎯 **Цель:** реализовать формулы дисперсии и стандартного отклонения в цикле.

#### 4.1. Отклонения от среднего

✏️ **Задание:** для списка `amounts = [5000, 12000, 8000, 15000, 7000]`:
- вычислите среднее арифметическое;
- для каждого элемента найдите отклонение от среднего (`amounts[i] - mean`);
- выведите отклонения на экран.

In [ ]:
## код пишите здесь

#### 4.2. Квадраты отклонений

✏️ **Задание:** для того же списка возведите каждое отклонение в квадрат. Выведите квадраты.

In [ ]:
## код пишите здесь

#### 4.3. Дисперсия

✏️ **Задание:** сложите все квадраты отклонений через цикл `for`. Разделите полученную сумму на `n − 1`. Выведите дисперсию с двумя знаками после запятой.

In [ ]:
## код пишите здесь

#### 4.4. Стандартное отклонение ($\sigma$)

✏️ **Задание:** извлеките квадратный корень из дисперсии — это и есть стандартное отклонение $\sigma$. Используйте `math.sqrt()`. Выведите $\sigma$ с двумя знаками после запятой.

💡 **Помните:** стандартное отклонение — это **корень из дисперсии**.


In [ ]:
## код пишите здесь

#### 4.5. Проверка через `statistics`

✏️ **Задание:** импортируйте модуль `statistics` через `import statistics` и вычислите:
- дисперсию через `statistics.variance(amounts)`;
- стандартное отклонение через `statistics.stdev(amounts)`.

Сравните результаты с 4.3 и 4.4.

🔍 **Вопрос для самопроверки:** почему мы делим на `n − 1`, а не на `n`? Что изменится, если разделить на `n`?

In [ ]:
## код пишите здесь

### Задание 5. Z-оценка и правило трёх сигм 

🎯 **Цель:** реализовать Z-оценку и найти аномалии.

#### 5.1. Z-оценки

✏️ **Задание:** для списка `amounts = [5000, 12000, 8000, 15000, 7000]`:
- вычислите среднее и $\sigma$;
- для каждого элемента вычислите Z-оценку по формуле `(x − mean) / $\sigma$`;
- выведите Z-оценки с двумя знаками после запятой.

In [ ]:
## код пишите здесь

#### 5.2. Аномалии по правилу трёх сигм

✏️ **Задание:** для списка `amounts = [5000, 12000, 8000, 15000, 7000, 200000]`:
- вычислите среднее и σ;
- найдите границы `[mean − 3·σ; mean + 3·σ]`;
- выведите все элементы за границами.

In [ ]:
## код пишите здесь

#### 5.3. Сравнение IQR и Z-оценки

✏️ **Задание:** для того же списка `amounts = [5000, 12000, 8000, 15000, 7000, 200000]`:
- найдите выбросы методом IQR (из лабораторной №2);
- найдите выбросы методом Z-оценки;
- сравните результаты.

Объясните, какой метод дал более адекватный результат и почему.

In [ ]:
## код пишите здесь

#### 5.4. Случай без выбросов

✏️ **Задание:** для списка `amounts_clean = [5000, 12000, 8000, 15000, 7000]` найдите аномалии по Z-оценке. Убедитесь, что их нет.

🔍 **Вопрос для самопроверки:** почему Z-оценка может «пропустить» выброс в маленькой выборке?

In [ ]:
## код пишите здесь

### Задание 6. Скользящее среднее 
🎯 **Цель:** освоить построение скользящего среднего.

#### 6.1. Скользящее среднее с окном 3

✏️ **Задание:** для списка `amounts = [5000, 5200, 5100, 5300, 5500, 5400, 5600]`:
- выберите окно `k = 3`;
- для каждой позиции начиная с 2-й возьмите последние 3 элемента;
- вычислите их среднее;
- сохраните результаты в список `ma`.

In [ ]:
## код пишите здесь

#### 6.2. Скользящее среднее с окном 5

✏️ **Задание:** для того же списка постройте скользящее среднее с окном `k = 5`.

In [ ]:
## код пишите здесь

#### 6.3. Сравнение сглаживания

✏️ **Задание:** сравните оригинальный список и скользящее среднее с окнами 3 и 5. Объясните, что происходит с колебаниями.

In [ ]:
## код пишите здесь

#### 6.4. Скользящее среднее на «шумных» данных

✏️ **Задание:** для списка `amounts = [5000, 15000, 3000, 12000, 4000, 18000, 6000]` (сильные колебания) постройте скользящее среднее с окном 3. Убедитесь, что сглаживание уменьшает разброс.

🔍 **Вопрос для самопроверки:** зачем нужно скользящее среднее? Что произойдёт, если окно слишком большое?

In [ ]:
## код пишите здесь

### Задание 7. Профильный кейс: автоматизированный скрининг выписки банка 

🎯 **Ситуация:** К вам поступила банковская выписка за месяц. Нужно автоматически:
- обработать все транзакции;
- найти аномалии двумя методами (IQR и Z-оценка);
- выявить тренд через скользящее среднее;
- сформулировать рекомендации.

#### 7.1. Данные

Возьмите данные своего варианта (см. раздел «Варианты данных»). Сохраните их в переменную `transactions`.

#### 7.2. Задачи

✏️ **Задание 7.2.1.** Выведите количество транзакций, сумму, среднее, медиану, $\sigma$.

✏️ **Задание 7.2.2.** Найдите выбросы методом IQR. Сколько их? Какие?

✏️ **Задание 7.2.3.** Найдите выбросы методом Z-оценки. Сколько их? Какие?

✏️ **Задание 7.2.4.** Сравните результаты двух методов. Какие транзакции признаны выбросами в обоих методах? Какие — только в одном?

✏️ **Задание 7.2.5.** Создайте новый список `clean` без выбросов (по IQR). Вычислите среднее и медиану без выбросов. Сравните с результатами 7.2.1.

✏️ **Задание 7.2.6.** Постройте скользящее среднее с окном 3 для исходной выписки. Сравните: есть ли тренд? Растут транзакции или падают?

✏️ **Задание 7.2.7.** Используя списковое включение, создайте список всех «крупных» транзакций (больше 50 000). Сколько их? Какие?

✏️ **Задание 7.2.8.** Найдите первую крупную транзакцию (больше 50 000) с помощью цикла и `break`. Выведите её индекс и значение.

#### 7.3. Оформление результатов

Создайте Markdown-ячейку и оформите результаты в виде таблицы:

| Показатель | С выбросами | Без выбросов |
|------------|-------------|--------------|
| Количество | | |
| Сумма | | |
| Среднее | | |
| Медиана | | |
| σ | | |
| Размах | | |

✏️ **Задание:** напишите список выявленных аномалий с указанием метода, которым каждая была найдена.

### Задание 8. Вывод 

🎯 **Цель:** сформулировать профессиональный вывод по результатам автоматизированного скрининга.

Создайте Markdown-ячейку и напишите вывод по шаблону:

## Вывод 

По результатам автоматизированного скрининга выписки (20 транзакций) выявлено:

— средний чек составляет ___ руб., медиана — ___ руб., σ — ___ руб.;
— методом IQR выявлено ___ выбросов;
— методом Z-оценки выявлено ___ выбросов;
— транзакции, признанные аномальными обоими методами: ___;
— наиболее подозрительная транзакция — ___ руб. (Z = ___);
— без учёта выбросов среднее снизилось с ___ до ___ руб.;
— скользящее среднее показывает ___ (рост / падение / стабильность).

Рекомендуется: ___.


**Требования к выводу:**
- 6–8 предложений.
- Содержание: что анализировалось, какие результаты, какие рекомендации.
- Стиль: деловой, профессиональный.

**Защита лабораторной:** студент должен объяснить, что делает каждая строка его кода.

## 6. Контрольные вопросы

1. Чем отличается цикл `for` от `while`?
2. Какие три формы записи у функции `range()`?
3. Зачем нужна функция `enumerate()`?
4. Что делает `break`? Что делает `continue`?
5. Что такое списковое включение? Приведите пример.
6. Что такое дисперсия? Как она связана со стандартным отклонением?
7. Что такое σ? Как его вычислить из дисперсии?
8. Почему мы делим на `n − 1`, а не на `n`?
9. Что такое Z-оценка? Что она показывает?
10. Как интерпретировать Z = 2,5? Z = −1,8?
11. Что такое правило трёх сигм?
12. Что такое скользящее среднее? Зачем оно нужно?
13. Как размер окна влияет на сглаживание?
14. В чём разница между IQR и Z-оценкой?
15. Какой метод устойчивее к выбросам? Почему?

## 7. Типичные ошибки и как их избежать

| Ошибка | Причина | Решение |
|--------|---------|---------|
| `IndexError` в цикле `for i in range(len(list))` | Ошибка в границах | Проверить `len()` |
| Бесконечный цикл `while` | Забыт счётчик | Использовать `for`, где возможно |
| `break` сработал не там | Неверное условие | Проверить через `print` |
| `continue` пропустил нужное | Условие слишком широкое | Сузить условие |
| Списковое включение не работает | Синтаксис | Проверить `[expr for x in data if cond]` |
| σ = 0 | Все значения одинаковы | Это нормально |
| Z-оценка не находит выброс | Маленькая выборка | Использовать IQR |
| Скользящее среднее короче списка | Окно больше данных | Проверить `k ≤ len(data)` |
| Деление на ноль | σ = 0 | Проверить `if sigma != 0` |

## 8. Полезные ссылки

| Ресурс | Ссылка |
|--------|--------|
| Документация Python: циклы | https://docs.python.org/3/tutorial/controlflow.html |
| Документация Python: списковые включения | https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions |
| Модуль statistics | https://docs.python.org/3/library/statistics.html |
| Дисперсия (обзор) | https://ru.wikipedia.org/wiki/Дисперсия_случайной_величины |
| Стандартное отклонение (обзор) | https://ru.wikipedia.org/wiki/Стандартное_отклонение |
| Скользящее среднее (обзор) | https://ru.wikipedia.org/wiki/Скользящее_среднее |


In [ ]:
## 10. Варианты данных

#Каждый вариант — это банковская выписка за месяц (20 транзакций). Номер варианта соответствует вашему номеру в журнале. В каждом варианте — 19 типичных транзакций (4 000 – 15 000 руб.) и один крупный выброс (в 50–70 раз больше).

### Вариант 1

transactions = [
    5000, 7200, 8100, 4900, 5300,
    6800, 7500, 9100, 4800, 5600,
    12000, 4700, 5200, 7800, 8400,
    9500, 6100, 5500, 6900, 500000
]


### Вариант 2

transactions = [
    6200, 5400, 7100, 8300, 4800,
    5900, 6600, 7700, 5100, 6400,
    8900, 5500, 6200, 7400, 6000,
    8100, 5700, 6800, 5200, 480000
]


### Вариант 3

transactions = [
    4500, 6800, 7200, 5600, 8100,
    5400, 6300, 4900, 7700, 6500,
    9200, 5800, 6100, 7300, 5900,
    8800, 5200, 6600, 7000, 520000
]


### Вариант 4

transactions = [
    5800, 7400, 6900, 5200, 8100,
    6300, 5600, 7800, 6400, 5700,
    8500, 6100, 7300, 5400, 6700,
    9300, 5900, 7200, 5500, 550000
]


### Вариант 5

transactions = [
    4900, 6100, 8300, 5700, 7400,
    5200, 6800, 7600, 5900, 6400,
    9100, 5300, 7000, 6200, 5500,
    8700, 5600, 7900, 6500, 470000
]

### Вариант 6

transactions = [
    6400, 5100, 7800, 5900, 7200,
    6800, 5500, 8300, 6100, 4900,
    9600, 5700, 7500, 6300, 5400,
    8900, 6000, 7100, 6600, 530000
]


### Вариант 7

transactions = [
    5500, 7900, 6200, 7100, 5800,
    4900, 8400, 6600, 5300, 7700,
    6000, 9200, 5600, 6900, 6400,
    8100, 5200, 7400, 5700, 490000
]


### Вариант 8

transactions = [
    6700, 5300, 8100, 5900, 7400,
    6200, 7800, 5100, 8600, 6500,
    5700, 7000, 6300, 5500, 8800,
    5400, 7600, 6100, 6900, 460000
]


### Вариант 9

transactions = [
    5200, 7000, 6400, 8500, 5800,
    7700, 6100, 5400, 7900, 6600,
    9100, 5900, 7200, 5500, 6800,
    8300, 5600, 7500, 6200, 510000
]


### Вариант 10

transactions = [
    6000, 7500, 5100, 8200, 6400,
    5700, 7900, 6800, 5300, 7100,
    8700, 5500, 6600, 7400, 5900,
    9400, 6200, 7700, 5400, 480000
]